# EDA — Ransomware Dataset 2024
**Dataset:** [mexwell/ransomware-dataset-2024](https://www.kaggle.com/datasets/mexwell/ransomware-dataset-2024) (Kaggle)

Objetivo: explorar o dataset (21.752 amostras — 10.876 maliciosas de 26 famílias de malware, 11 delas de ransomware, e 10.876 benignas) para preparar as fases seguintes do CRISP-ML (Data Preparation e Modelação).

Este notebook segue o roadmap:
1. Setup e carregamento dos dados
2. Visão geral (shape, tipos, missing, duplicados)
3. Análise univariada
4. Análise bivariada
5. Análise multivariada
6. Análise de correlação (Pearson / Spearman)
7. EDA avançada (skewness, kurtosis, outliers)
8. Conclusões e próximos passos


## 1. Setup

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
%matplotlib inline


## 2. Carregar os dados

> **Nota:** descarrega o dataset do Kaggle (`mexwell/ransomware-dataset-2024`) e ajusta o caminho abaixo para o(s) ficheiro(s) CSV que vieres a extrair. Se o dataset vier com mais do que um ficheiro (ex: um por classe/família), junta-os num único DataFrame antes de continuar.

In [ ]:
# Ajusta o caminho para o(s) teu(s) ficheiro(s)
DATA_PATH = "ransom.csv"

df = pd.read_csv(DATA_PATH)

# O dataset mistura números decimais, valores hexadecimais e texto descritivo.
# Converte apenas colunas em que a maioria dos valores pode ser interpretada.
def parse_numeric_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"0x[0-9a-fA-F]+", text)
    if match:
        return int(match.group(0), 16)
    try:
        return float(text)
    except ValueError:
        return np.nan

for col in df.columns:
    if df[col].dtype == "object":
        parsed = df[col].map(parse_numeric_value)
        if parsed.notna().mean() >= 0.90:
            df[col] = parsed
print(df.shape)
df.head()


## 3. Visão geral do dataset

In [ ]:
df.info()


In [ ]:
# Valores em falta
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Colunas com valores em falta:")
missing


In [ ]:
# Duplicados
n_dup = df.duplicated().sum()
print(f"Linhas duplicadas: {n_dup}")


In [ ]:
TARGET_COL = "Class"
FAMILY_COL = "Family"


print(df[TARGET_COL].value_counts())


## 4. Análise univariada

In [ ]:
# Distribuição da classe (malicioso vs. benigno)
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x=TARGET_COL)
plt.title("Distribuição da classe (malicioso vs. benigno)")
plt.show()


In [ ]:
# Distribuição por família de malware (se aplicável)
if FAMILY_COL in df.columns:
    plt.figure(figsize=(10, 5))
    df[FAMILY_COL].value_counts().plot(kind="bar")
    plt.title("Distribuição por família de malware")
    plt.ylabel("Nº de amostras")
    plt.xticks(rotation=75)
    plt.show()


In [ ]:
# Histogramas das features numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in [TARGET_COL]]

df[num_cols].hist(figsize=(16, 12), bins=30)
plt.tight_layout()
plt.show()


In [ ]:
# Estatística descritiva
df[num_cols].describe().T


## 5. Análise bivariada (feature vs. classe)

In [ ]:
# Boxplots das features mais relevantes por classe
# Ajusta a lista consoante as colunas reais do dataset
features_relevantes = num_cols[:8]  # começa pelas primeiras 8 e ajusta

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), features_relevantes):
    sns.boxplot(data=df, x=TARGET_COL, y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
# Teste estatístico simples: diferença de médias entre classes (t-test) para cada feature numérica
resultados = []
classes = df[TARGET_COL].unique()
if len(classes) == 2:
    g0 = df[df[TARGET_COL] == classes[0]]
    g1 = df[df[TARGET_COL] == classes[1]]
    for col in num_cols:
        stat, p = stats.ttest_ind(g0[col].dropna(), g1[col].dropna(), equal_var=False)
        resultados.append({"feature": col, "t_stat": stat, "p_value": p})

    resultados_df = pd.DataFrame(resultados).sort_values("p_value")
    resultados_df


## 6. Análise multivariada

In [ ]:
# Pairplot de um subconjunto de features (pairplot com muitas colunas fica ilegível/lento)
subset_cols = num_cols[:5] + [TARGET_COL]
sns.pairplot(df[subset_cols], hue=TARGET_COL, corner=True, plot_kws={"alpha": 0.4, "s": 15})
plt.show()


In [ ]:
# PCA para visualizar a separabilidade das classes em 2D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df[num_cols].fillna(df[num_cols].median())
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=df[TARGET_COL], alpha=0.5, s=15)
plt.title(f"PCA (var. explicada: {pca.explained_variance_ratio_.sum():.2%})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


## 7. Análise de correlação

In [ ]:
# Correlação de Pearson
corr_pearson = df[num_cols].corr(method="pearson")

plt.figure(figsize=(14, 12))
sns.heatmap(corr_pearson, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Matriz de correlação — Pearson")
plt.show()


In [ ]:
# Correlação de Spearman (relações monótonas/não-lineares)
corr_spearman = df[num_cols].corr(method="spearman")

plt.figure(figsize=(14, 12))
sns.heatmap(corr_spearman, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Matriz de correlação — Spearman")
plt.show()


In [ ]:
# Pares de features altamente correlacionadas (candidatas a remover no Data Preparation)
threshold = 0.9
corr_abs = corr_pearson.abs()
pares_altos = (
    corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
pares_altos.columns = ["feature_1", "feature_2", "correlacao"]
pares_altos = pares_altos[pares_altos["correlacao"] > threshold].sort_values("correlacao", ascending=False)
pares_altos


## 8. EDA avançada: assimetria, curtose e outliers

In [ ]:
# Skewness e Kurtosis de cada feature numérica
assimetria = df[num_cols].skew().sort_values(ascending=False)
curtose = df[num_cols].kurtosis().sort_values(ascending=False)

resumo_forma = pd.DataFrame({"skewness": assimetria, "kurtosis": curtose})
resumo_forma


In [ ]:
# Deteção de outliers via IQR
def contar_outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((serie < limite_inf) | (serie > limite_sup)).sum()

outliers = {col: contar_outliers_iqr(df[col].dropna()) for col in num_cols}
outliers_df = pd.Series(outliers, name="n_outliers").sort_values(ascending=False)
outliers_df.head(20)


## 9. Conclusões e próximos passos

Preenche esta secção depois de correr o notebook com os dados reais:

- **Qualidade dos dados:** (missing values, duplicados, tipos a corrigir)
- **Balanceamento:** confirmar se a distribuição de classes/famílias se mantém equilibrada
- **Features mais discriminativas:** (com base no t-test e nos boxplots)
- **Features redundantes a remover:** (pares com correlação > 0.9)
- **Necessidade de normalização/transformação:** (com base em skewness/kurtosis e outliers)
- **Decisão sobre o problema:** classificação binária (malicioso vs. benigno) ou multiclasse (família)

Estas conclusões alimentam diretamente a fase de **Data Preparation** do roadmap.
